<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:46px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Medical Computer Vision · Anatomical Prior · Trustworthy Deep Learning</div>
  <h1 style="font-size:2.05em;margin:0 0 12px 0;font-weight:700;line-height:1.25; color:#f0f6fc !important;">Akciğer-Odaklı Vision Transformer:<br>Anatomik Maskeleme ile Göğüs Radyografisinde Pnömoni Sınıflandırması</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5; color:#f0f6fc !important;">Kısayolları Cezalandırmak Yerine Yok Etmek: Segmentasyon-Güdümlü Girdi Maskeleme ile <i>Right-for-the-Right-Reasons</i> Mimarisi</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Veri:</b> Chest X-Ray Pneumonia (Balanced) — PA/AP CXR</div>
    <div><b>Sınıflandırıcı:</b> ViT-B/16 (ImageNet ön-eğitimli, son katmanlar ince ayar)</div>
    <div><b>Anatomik Önsel:</b> <code>ianpan/chest-x-ray-basic</code> (U-Net; akciğer Dice R=0.957 / L=0.948)</div>
    <div><b>Yöntem:</b> Girdi-seviyesi akciğer maskeleme + RoI kırpma (offline)</div>
    <div><b>Açıklanabilirlik:</b> Attention Rollout (CLS→patch akışı)</div>
    <div><b>Doğrulama:</b> Lung-Focus Ratio (LFR) — akciğer-içi odak metriği</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Tez:</b> Modele "akciğer dışına <i>bakma</i>" demek (yumuşak RRR cezası) zayıf bir kısıttır; model bu cezayı sayısal olarak küçültürken kararını hâlâ akciğer-dışı sinyale (omurga/mediasten/postür) dayandırabilir. Bu çalışma daha sert bir önsel uygular: akciğer dışındaki <i>tüm</i> pikseller, sınıflandırıcı görüntüyü hiç görmeden önce silinir. Böylece kenar artefaktları, R/L işaretleyicileri, omurga ve mediasten ekseni <b>fiziksel olarak</b> ortadan kaldırılır ve model yalnızca akciğer parankimine bakmak <b>zorunda</b> bırakılır.
  </div>
</div>

## Öz (Abstract)

Vision Transformer (ViT) modelleri göğüs radyografisinde (CXR) pnömoni sınıflandırmasında çok yüksek test metrikleri üretebilir; ancak bu başarım, kararın gerçek patolojiye mi yoksa veri kümesindeki yapısal yanlılıklara (çekim protokolü, kenar işaretleyicileri, hasta postürü) mı dayandığını söylemez. Bu olgu **kısayol öğrenmesi** (shortcut learning / Clever Hans etkisi) olarak bilinir.

Bu çalışmanın önceki sürümü, kısayolları bir **açıklanabilirlik denetimi** ile teşhis etmiş ve modele akciğer dışına bakmayı cezalandıran bir *Right-for-the-Right-Reasons (RRR)* terimi uygulamıştı. Sonuç öğreticiydi: model köşe kısayolunu büyük ölçüde bıraksa da dikkati ısrarla **iki akciğer arasındaki omurga/mediasten eksenine** kaymış, akciğer-odak oranı (LFR) yalnızca **≈ 0.16** düzeyinde kalmıştı. Bu, **yumuşak bir cezanın güçlü bir kısayolu caydırmaya yetmediğinin** doğrudan kanıtıdır.

Bu sürümde yaklaşım değiştirilir: kısayolu *cezalandırmak* yerine, onu besleyen bilgi **girdi seviyesinde yok edilir.** Her radyograf, ViT'e ulaşmadan önce U-Net tabanlı bir segmentasyon ağıyla (`ianpan/chest-x-ray-basic`) işlenir; **yalnızca sağ ve sol akciğer alanları** korunur (kalp, mediasten, diyafram, köşe artefaktları, R/L harfleri ve gövde dokusu silinir), ardından akciğer bölgesi bir **İlgi Alanı (RoI)** kutusuna kırpılıp $224\times224$'e ölçeklenir. Bu sayede ViT, anatomik olarak akciğer dışına bakamaz hale gelir — *Right for the Right Reasons* artık bir ceza terimiyle değil, **mimarinin yapısıyla** garanti altına alınır.

Segmentasyonun kusurlu olabileceği (akciğer Dice R=0.957, L=0.948) göz önünde tutularak, sınır patolojilerinin (periferik konsolidasyon, kostofrenik açı küntleşmesi) kaybını önlemek için maske **genişletme (dilation)**, **kenar yumuşatma (feathering)** ve **RoI güvenlik payı** uygulanır. Eğitim sonrası Attention Rollout ve LFR ile modelin dikkatinin akciğer-içi odağı niceliksel olarak doğrulanır.

---

## İçindekiler

| § | Bölüm | Odak |
|---|-------|------|
| 1 | [Giriş ve Yöntemin Gerekçesi](#1) | Yumuşak cezadan sert önsele geçiş |
| 2 | [Anatomik Segmentasyon Modeli](#2) | ianpan U-Net, çıktı yapısı |
| 3 | [Akciğer-Odaklı Maskeleme + RoI Kırpma](#3) | Çekirdek dönüşüm, tolerans tasarımı |
| 4 | [Önizleme: Maskeleme Etkisi](#4) | Orijinal → maske → kırpım |
| 5 | [Veri Bölümleme ve Sızıntı Denetimi](#5) | Dengeli val/test, leakage |
| 6 | [Offline Ön-İşleme (Önbellekleme)](#6) | Tüm seti bir kez maskeleyip diske yaz |
| 7 | [Veri Yükleyiciler ve Artırma](#7) | Maskeli görüntüye uygun augmentation |
| 8 | [ViT-B/16 Mimarisi ve Eğitim](#8) | Saf CrossEntropy (RRR yok) |
| 9 | [Model Değerlendirmesi](#9) | Confusion / ROC / PR |
| 10 | [Açıklanabilirlik: Attention Rollout](#10) | Akciğer-içi lokalizasyon |
| 11 | [Akciğer-Odak Doğrulaması (LFR)](#11) | Dürüst yorum, kısıtlar |
| 12 | [Tartışma ve Sonuç](#12) | Önceki yöntemle kıyas, kısıtlar |

---

## 1. Giriş ve Yöntemin Gerekçesi <a id='1'></a>

### 1.1. Problem: Yüksek Doğruluk ≠ Klinik Güvenilirlik

Pnömoni, göğüs radyografisinde akciğer parankimindeki **konsolidasyon ve infiltrasyon** olarak görünür. Derin öğrenme modelleri bu görevde yüksek doğruluk raporlar; ancak bir model, doğru etiketi patolojiyle nedensel ilişkisi olmayan istatistiksel ipuçlarından (çekim cihazı, hastane, postür, kenar işaretleyicileri) da tahmin edebilir. Tıbbi görüntülemede bu olgunun klasik örneği DeGrave ve ark. (2021) çalışmasıdır: "COVID-19 tespiti" yaptığı sanılan modellerin aslında görüntü kaynağına ve pozisyonel artefaktlara baktığı gösterilmiştir.

### 1.2. Önceki Bulgu: Yumuşak Ceza Yetersizdir

Bu projenin önceki sürümü kısayolları üç fazda soydu: (1) köşe **L/R işaretleyicileri** ve PA/AP kenar dokusu → merkezi kırpma + köşe-token cezasıyla elendi; (2) klasik (Otsu) segmentasyonun pnömoni infiltrasyonunu maske dışına itme yanılgısı → **U-Net segmentasyonuna** geçildi; (3) eğitime akciğer-dışı token'ları cezalandıran bir **RRR (anatomik bastırım) terimi** eklendi.

Buna rağmen nihai analiz çarpıcıydı: küresel **LFR yalnızca $0.159 \pm 0.125$** çıktı ve dikkat ısrarla **omurga/mediasten ekseninde** yoğunlaştı. Yani model köşelere bakmayı bıraksa da, *iki akciğer arasındaki santral bölgeden* — büyük olasılıkla **AP/PA postür yanlılığını** sömürerek — karar veriyordu. Ceza terimi $\lambda=5.0$ altında $\sim 3\times10^{-4}$ düzeyine inse de bu güçlü kısayolu caydıramadı. **Ders:** modele "bakma" demek (yumuşak, türevlenebilir kısıt), model o bölgeyi hâlâ *görebildiği* sürece dolanılabilir bir kısıttır.

### 1.3. Bu Çalışma: Cezalandırma Değil, Yok Etme

Mantıksal sonuç açıktır: eğer akciğer-dışı bilgi modelin **girdisinde hiç bulunmazsa**, model onu sömüremez. Bu çalışma kısayolu besleyen sinyali kaynağında keser:

$$\text{Orijinal CXR} \;\xrightarrow{\text{ianpan U-Net}}\; \text{akciğer maskesi} \;\xrightarrow{\text{maske + RoI kırp}}\; \text{yalnızca akciğer} \;\xrightarrow{\text{Resize}(224)}\; \text{ViT}$$

Bu, *Right for the Right Reasons* ilkesini bir kayıp terimine değil, **veri hattının yapısına** gömer. Köşe artefaktları, R/L harfleri, gövde dokusu, **omurga ve mediasten** — hepsi ViT'e ulaşmadan silinir.

### 1.4. Tasarımın Dürüst Kısıtları

Bu yaklaşım güçlüdür ama bedelsiz değildir; baştan açıkça belirtilir:

1. **Segmentasyon bağımlılığı.** Maske yanlışsa girdi de yanlış olur. ianpan akciğer Dice'ı R=0.957 / L=0.948'dir — yani ~%5 sınır hatası beklenir. Periferik patoloji (kostofrenik açıda efüzyon, plevral temaslı konsolidasyon) maske dışında kalabilir. Bunu telafi için maske **genişletilir**, kenarlar **yumuşatılır** ve RoI'ye **güvenlik payı** eklenir (§3).
2. **Bağlam kaybı.** Radyolog kardiyotorasik oran, mediastinal genişlik gibi akciğer-dışı bulguları da kullanır; bu yöntem onları kasıtlı olarak atar. Amaç tam tanı değil, **kısayolsuz** parankim-temelli sınıflandırmadır.
3. **LFR'nin zayıflaması.** Girdi zaten yalnızca akciğerden ibaret olduğundan, "model akciğere bakıyor mu?" sorusu kısmen totolojiktir. LFR artık "dikkat *görünür akciğer* üzerinde mi, yoksa maskelenmiş kenarda mı israf ediliyor?" sorusunu ölçer (§11'de açıkça tartışılır).

### 1.5. Deneysel Ortam

Aşağıdaki hücre, yeniden üretilebilirlik için sabit tohum, aygıt ve tüm hiperparametreleri (`CONFIG`) tanımlar. Maskeleme toleransı parametreleri (`mask_dilate_frac`, `roi_pad_frac`, `mask_feather`) burada merkezîleştirilmiştir.

In [ ]:
import os, warnings, random, time, types
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             roc_curve, auc, precision_recall_curve, average_precision_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*52)
print(f"  Cihaz : {device}")
if device.type == 'cuda':
    print(f"  GPU   : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("="*52)

CONFIG = {
    "img_size"       : 224,          # ViT-B/16 sabit girdi
    "patch_size"     : 16,           # 224/16 -> 14x14 = 196 patch
    "batch_size"     : 16,
    "epochs"         : 15,
    "lr"             : 1e-4,
    "weight_decay"   : 1e-2,
    "dropout"        : 0.1,
    "num_classes"    : 2,
    "mean"           : [0.4769, 0.4769, 0.4769],
    "std"            : [0.2414, 0.2414, 0.2414],

    # ---- Anatomik maskeleme / RoI hattı (ianpan Dice R=0.957 / L=0.948 toleransı) ----
    "orig_short_max"   : 512,    # segmentasyon icin orijinalin kisa kenarini bu degere indir (hiz)
    "mask_dilate_frac" : 0.025,  # maskeyi kisa-kenarin %2.5'i kadar genislet -> sinir patolojisini koru
    "roi_pad_frac"     : 0.05,   # RoI kutusuna %5 guvenlik payi
    "mask_feather"     : 9,      # maske kenari Gaussian yumusatma (tek sayi) -> sert kenar artefaktini onle
    "fill_mode"        : "mean", # akciger disi dolgu: "mean" (veri ortalamasi gri) veya "black"
    "cache_root"       : "/kaggle/working/lung_focused",  # maskelenmis goruntu onbellegi
}

BASE_PATH  = "/kaggle/input/datasets/yusufmurtaza01/chest-xray-pneumonia-balanced-dataset"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH   = os.path.join(BASE_PATH, "val")
TEST_PATH  = os.path.join(BASE_PATH, "test")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

CLASS_COLORS = {'NORMAL': '#2E8B57', 'PNEUMONIA': '#DC143C'}
MEDICAL_CMAP = LinearSegmentedColormap.from_list(
    'medical', ['#000033', '#0033CC', '#00CCFF', '#FFFF00', '#FF4500'], N=256)

print(f"\nHazir | PyTorch {torch.__version__}")

## 2. Anatomik Segmentasyon Modeli <a id='2'></a>

Akciğer maskesi `ianpan/chest-x-ray-basic` modelinden alınır. Bu, CheXmask veri kümesi (335K+ görüntü) üzerinde eğitilmiş bir U-Net'tir ve modelin kendi raporladığı segmentasyon başarımı yüksektir: **Sağ akciğer Dice = 0.957, Sol akciğer Dice = 0.948, Kalp Dice = 0.943.**

Model çoklu çıktı verir; bizi ilgilendiren `out["mask"]` tensörüdür. Kanal boyutunda `argmax` alındığında her piksel şu etiketlerden birini alır:

| Değer | Anatomi |
|-------|---------|
| 0 | Arka plan |
| **1** | **Sağ akciğer** |
| **2** | **Sol akciğer** |
| 3 | Kalp |

Bu çalışmada **yalnızca 1 ve 2** (sağ + sol akciğer) korunur. **Kalbin (3) kasıtlı olarak dışlanması kritiktir:** önceki sürümde keşfedilen omurga/mediasten kısayolu tam da iki akciğer arasındaki bu santral bölgede yaşıyordu; onu maskenin dışında bırakmak kısayolu kaynağında yok eder.

> **Not.** `AutoModel.from_pretrained` çağrısı, bazı `transformers` sürümlerinde `all_tied_weights_keys` özniteliği nedeniyle hata verebilir. Aşağıdaki geçici yama (monkey-patch) yalnızca model yükleme anında devreye girer ve hemen geri alınır; ana ViT modelimizin yapısını etkilemez.

In [ ]:
import transformers
from transformers import AutoModel

print("ianpan/chest-x-ray-basic segmentasyon modeli yukleniyor...")

# --- transformers surum uyumu icin gecici yama ---
_orig_finalize = transformers.modeling_utils.PreTrainedModel._finalize_model_loading
def _safe_finalize(model, *a, **k):
    if not hasattr(model, 'all_tied_weights_keys'):
        model.all_tied_weights_keys = {}
    return _orig_finalize(model, *a, **k)
transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _safe_finalize

try:
    seg_model = AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                          trust_remote_code=True).to(device).eval()
    print("Segmentasyon modeli hazir (Dice: R=0.957, L=0.948, Kalp=0.943).")
finally:
    # ViT modeli bu yamadan etkilenmesin diye geri al
    transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _orig_finalize

## 3. Akciğer-Odaklı Maskeleme + RoI Kırpma <a id='3'></a>

Çekirdek dönüşüm üç adımdan oluşur ve diğer modelin uyardığı tüm teknik riskleri tek tek karşılar:

1. **Akciğer maskesi (ianpan).** Orijinal (kırpılmamış) görüntüde sağ + sol akciğer çıkarılır. Segmentasyon hız için kısa kenarı `orig_short_max` (512) ile sınırlı bir görüntüde yapılır; bu çözünürlük akciğer sınırları için fazlasıyla yeterlidir.
2. **Tolerans katmanı (Dice ≈ 0.95 telafisi).**
   - *Genişletme (dilation):* Maske, kısa kenarın `mask_dilate_frac`'ı (≈ %2.5) kadar morfolojik olarak büyütülür. ~%5'lik sınır hatası, böylece akciğer kenarındaki patolojiyi (periferik konsolidasyon, kostofrenik efüzyon) dışarıda bırakmaz.
   - *Kenar yumuşatma (feathering):* Maske kenarına Gauss bulanıklığı uygulanır. Bu, akciğer ↔ arka plan geçişindeki **ani (hard) yoğunluk sıçramasını** giderir; aksi halde ViT bu keskin yapay sınırı yeni bir "özellik" sanabilirdi.
   - *Dolgu (fill):* Akciğer dışı, saf siyah (0) yerine **veri kümesi ortalama grisiyle** doldurulur. Normalizasyon sonrası bu değer ≈ 0 olduğundan, siyahın yarattığı güçlü yapay kenara kıyasla çok daha nötrdür.
3. **RoI kırpma + ölçekleme.** Maskenin sınır kutusu (bounding box) bulunur, `roi_pad_frac` (%5) payla genişletilir, **kareye** tamamlanır (en-boy oranı korunsun) ve $224\times224$'e ölçeklenir. Böylece devasa boş kenarlar atılır; akciğerler kareyi doldurur.

**Geri çekilme (fallback).** Segmentasyon boş maske dönerse (nadir hata), o görüntü için orijinal doğrudan 224'e ölçeklenir ve `used=False` ile işaretlenir; istatistik raporunda ayrıca sayılır.

In [ ]:
def load_image(path, short_max):
    """Orijinali yukler; cok buyukse kisa kenari short_max'a indirir. (rgb_u8, gray_u8) doner."""
    pil = Image.open(path).convert('RGB')
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        W0, H0 = int(round(W0 * s)), int(round(H0 * s))
        pil = pil.resize((W0, H0), Image.BILINEAR)
    rgb  = np.asarray(pil).astype(np.uint8)        # (H,W,3)
    gray = np.asarray(pil.convert('L'))            # (H,W) uint8
    return rgb, gray


@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    """ianpan ile sag+sol akciger maskesi (kalp haric). out_hw=(H,W) -> (H,W) uint8 {0,1}."""
    x = seg_model.preprocess(gray_u8)                       # ianpan: kareye resize + normalize
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)  # (1,1,h,w)
    logits = seg_model(x)["mask"]                           # (1,C,h,w)
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()            # (H,W) in {0,1,2,3}
    lung = ((pred == 1) | (pred == 2)).astype(np.uint8)     # 1=sag, 2=sol; 3=kalp DISLANIR
    return lung


def make_lung_focused(rgb_u8, lung_u8, cfg):
    """Maske uygula + RoI kirp + 224'e olcekle.
    Doner: (img224_u8 RGB, mask224_u8 {0,1}, used_bool)."""
    H, W = lung_u8.shape
    short = min(H, W)
    S = cfg["img_size"]

    # --- Segmentasyon basarisiz: orijinali olceklendir (fallback) ---
    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        return fb, np.ones((S, S), np.uint8), False

    # --- 1) Tolerans: maskeyi genislet ---
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*dil+1, 2*dil+1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)        # (H,W) {0,1}

    # --- 2) Yumusak kenar (sert maskeleme artefaktini onle) ---
    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0: f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]               # (H,W,1)

    # --- 3) Akciger disini doldur ---
    if cfg["fill_mode"] == "mean":
        fill = np.array([m*255.0 for m in cfg["mean"]], dtype=np.float32)
    else:
        fill = np.zeros(3, dtype=np.float32)
    masked = rgb_u8.astype(np.float32) * soft + fill[None, None, :] * (1.0 - soft)
    masked = masked.clip(0, 255).astype(np.uint8)

    # --- 4) RoI sinir kutusu + pay, kareye genislet ---
    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)

    bh, bw = (y1 - y0 + 1), (x1 - x0 + 1)
    side = min(max(bh, bw), min(H, W))                      # kareyi goruntuye sigdir
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side);     tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side);     tx0 = max(0, tx1 - side)  # kenara tasarsa geri it

    roi      = masked[ty0:ty1, tx0:tx1]
    roi_mask = mask_d[ty0:ty1, tx0:tx1]

    img224  = cv2.resize(roi, (S, S), interpolation=cv2.INTER_AREA)
    mask224 = cv2.resize(roi_mask, (S, S), interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    return img224, mask224, True

print("Maskeleme + RoI fonksiyonlari hazir.")

## 4. Önizleme: Maskeleme Etkisi <a id='4'></a>

Hatta yatırım yapmadan önce dönüşümü her sınıftan birkaç örnekte gözle doğrularız. Her satır: **(1) orijinal radyograf + akciğer konturu**, **(2) maske uygulanmış görüntü**, **(3) RoI'ye kırpılmış nihai ViT girdisi**. Kalbin ve omurganın iki akciğer arasındaki santral bölgede **maske dışında** kaldığına dikkat edin — bu, önceki sürümdeki kısayolun yaşadığı bölgedir.

In [ ]:
def _preview_paths(n=3):
    sel = {}
    for cls in ['NORMAL', 'PNEUMONIA']:
        d = os.path.join(TEST_PATH, cls)
        files = sorted(os.listdir(d))[:n]
        sel[cls] = [os.path.join(d, f) for f in files]
    return sel

_sel = _preview_paths(n=3)
rows = sum(len(v) for v in _sel.values())
fig, axes = plt.subplots(rows, 3, figsize=(11, 3.6*rows))
fig.suptitle('Akciger-Odakli Maskeleme: Orijinal -> Maske -> RoI Kirpim (ViT girdisi)',
             fontsize=12, fontweight='bold', y=1.0)

r = 0
for cls in ['NORMAL', 'PNEUMONIA']:
    for path in _sel[cls]:
        rgb, gray = load_image(path, CONFIG["orig_short_max"])
        lung = lung_mask_ianpan(gray, gray.shape)
        img224, mask224, used = make_lung_focused(rgb, lung, CONFIG)

        ax = axes[r, 0]
        ax.imshow(rgb)
        ax.contour(lung, levels=[0.5], colors='lime', linewidths=1.4)
        ax.set_title(f'{cls} — Orijinal + akciger konturu\n(akciger alan={lung.mean():.0%})',
                     fontsize=9, color=CLASS_COLORS[cls], fontweight='bold')
        ax.axis('off')

        # maskeli (kirpimsiz) gosterim
        soft = lung.astype(np.float32)
        if CONFIG["mask_feather"] >= 3:
            soft = cv2.GaussianBlur(soft, (CONFIG["mask_feather"], CONFIG["mask_feather"]), 0)
        fill = np.array([m*255 for m in CONFIG["mean"]]) if CONFIG["fill_mode"]=="mean" else np.zeros(3)
        mv = (rgb.astype(np.float32)*soft[...,None] + fill*(1-soft[...,None])).clip(0,255).astype(np.uint8)
        ax = axes[r, 1]; ax.imshow(mv)
        ax.set_title('Maske uygulandi (akciger disi silindi)', fontsize=9); ax.axis('off')

        ax = axes[r, 2]; ax.imshow(img224)
        tag = '' if used else '  [FALLBACK]'
        ax.set_title(f'Nihai 224x224 ViT girdisi{tag}', fontsize=9, color='navy', fontweight='bold')
        ax.axis('off')
        r += 1

plt.tight_layout()
plt.savefig('fig_01_masking_preview.png', dpi=130, bbox_inches='tight')
plt.show()
print("Onizleme hazir: akciger disi (omurga/mediasten/kose dahil) silindi.")

## 5. Veri Bölümleme ve Sızıntı Denetimi <a id='5'></a>

Orijinal doğrulama kümesi çok küçük olduğundan (~30 örnek) F1 tahminini güvenilmez kılar. Bu nedenle **val + test havuzu birleştirilip** her sınıftan eşit sayıda örnek içeren tam dengeli iki alt kümeye bölünür. Ayrıca dosya imzası (ad + boyut) ile **train ↔ val/test** ve **val ↔ test** çakışmaları denetlenerek **veri sızıntısı (data leakage)** engellenir — raporlanan metriklerin geçerliliği için zorunludur.

Bu aşamada yalnızca **(dosya yolu, etiket)** listeleri oluşturulur; maskeleme bir sonraki adımda (offline) yapılır.

In [ ]:
def file_sig(p):
    return f"{os.path.basename(p)}_{os.path.getsize(p) if os.path.exists(p) else 0}"

train_if = datasets.ImageFolder(TRAIN_PATH)
val_if   = datasets.ImageFolder(VAL_PATH)
test_if  = datasets.ImageFolder(TEST_PATH)

CLASS_NAMES  = train_if.classes                    # ['NORMAL','PNEUMONIA']
CLASS_TO_IDX = train_if.class_to_idx
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

train_samples = list(train_if.samples)
pool = list(val_if.samples) + list(test_if.samples)
print("="*62)
print(f"  Havuz: Val({len(val_if.samples)}) + Test({len(test_if.samples)}) = {len(pool)}")

# --- Sizinti: train ile cakisan dosyalari havuzdan cikar ---
train_sigs = set(file_sig(p) for p, _ in train_samples)
before = len(pool)
pool = [(p, l) for p, l in pool if file_sig(p) not in train_sigs]
leak = before - len(pool)
print(f"  Train sizintisi: {leak} dosya cikarildi." if leak else "  Train sizintisi yok.")

# --- Dengeli bolme ---
idx_n, idx_p = CLASS_TO_IDX['NORMAL'], CLASS_TO_IDX['PNEUMONIA']
normal_pool = [(p, l) for p, l in pool if l == idx_n]
pneum_pool  = [(p, l) for p, l in pool if l == idx_p]
random.shuffle(normal_pool); random.shuffle(pneum_pool)
per = min(len(normal_pool), len(pneum_pool)) // 2
print(f"  Havuzda NORMAL={len(normal_pool)}, PNEUMONIA={len(pneum_pool)} | her sette sinif basina={per}")

val_samples  = normal_pool[:per]      + pneum_pool[:per]
test_samples = normal_pool[per:2*per] + pneum_pool[per:2*per]
random.shuffle(val_samples); random.shuffle(test_samples)

splits_raw = {'train': train_samples, 'val': val_samples, 'test': test_samples}

# --- Capraz sizinti kontrolu ---
vp = set(p for p, _ in val_samples); tp = set(p for p, _ in test_samples)
trp = set(p for p, _ in train_samples)
print("="*62)
from collections import Counter
for name, s in splits_raw.items():
    c = Counter(l for _, l in s); tot = len(s)
    print(f"  {name.upper():<6}: {tot:>5}  | NORMAL:{c[idx_n]:>4}  PNEUMONIA:{c[idx_p]:>4}")
print("-"*62)
print(f"  Val<->Test cakisma : {len(vp & tp)}")
print(f"  Train<->Val/Test   : {len(trp & (vp | tp))}")
print("="*62)

## 6. Offline Ön-İşleme (Önbellekleme) <a id='6'></a>

Segmentasyon ağını her epoch'ta `__getitem__` içinde anlık (on-the-fly) çalıştırmak eğitimi günlerce uzatır (her görüntü için ileri-geçiş). Bu nedenle **offline ön-işleme** tercih edilir: tüm görüntüler **bir kez** maskelenip kırpılır ve diske yazılır. Eğitim, ardından bu hafif, hazır 224×224 görüntüler üzerinde standart hızda çalışır.

Her görüntü için iki çıktı önbelleğe alınır:
- `cache_root/{split}/{class}/{idx_ad}.png` — nihai ViT girdisi (maskeli + kırpılmış).
- `cache_root/{split}_mask/{class}/{idx_ad}.png` — aynı uzaydaki ikili akciğer maskesi (LFR doğrulaması §11 için).

Dosya adlarına benzersiz indeks öneki eklenir (olası ad çakışmalarını önler). `overwrite=False` ise mevcut dosyalar atlanır; hücre güvenle yeniden çalıştırılabilir.

> **Süre.** ~8.500 görüntü × segmentasyon ≈ tek seferlik birkaç dakikadır (T4); sonuç önbelleğe alındığından sonraki tüm çalıştırmalar anlıktır.

In [ ]:
CACHE_ROOT = CONFIG["cache_root"]

def preprocess_split(name, samples, cfg, overwrite=False, log_every=500):
    img_dir  = {c: os.path.join(CACHE_ROOT, name, c)          for c in CLASS_NAMES}
    mask_dir = {c: os.path.join(CACHE_ROOT, f"{name}_mask", c) for c in CLASS_NAMES}
    for c in CLASS_NAMES:
        os.makedirs(img_dir[c], exist_ok=True)
        os.makedirs(mask_dir[c], exist_ok=True)

    n_ok = n_fb = n_err = 0
    t0 = time.time()
    for i, (path, label) in enumerate(samples):
        cls  = IDX_TO_CLASS[label]
        stem = f"{i:06d}_{os.path.splitext(os.path.basename(path))[0]}.png"
        out_img  = os.path.join(img_dir[cls],  stem)
        out_mask = os.path.join(mask_dir[cls], stem)
        if (not overwrite) and os.path.exists(out_img) and os.path.exists(out_mask):
            n_ok += 1; continue
        try:
            rgb, gray = load_image(path, cfg["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            img224, mask224, used = make_lung_focused(rgb, lung, cfg)
            Image.fromarray(img224).save(out_img)
            Image.fromarray((mask224*255).astype(np.uint8)).save(out_mask)
            n_ok += 1; n_fb += (0 if used else 1)
        except Exception as e:
            n_err += 1
            if n_err <= 5:
                print(f"  hata ({os.path.basename(path)}): {e}")
        if (i+1) % log_every == 0:
            print(f"  [{name}] {i+1}/{len(samples)}  ({time.time()-t0:.0f}s)")
    print(f"[{name}] tamam: {n_ok} kayit | fallback={n_fb} | hata={n_err} | {time.time()-t0:.0f}s")

print("Offline maskeleme + kirpma basliyor...\n")
for name, samples in splits_raw.items():
    preprocess_split(name, samples, CONFIG, overwrite=False)
print("\nOnbellek hazir:", CACHE_ROOT)

## 7. Veri Yükleyiciler ve Artırma <a id='7'></a>

Girdiler artık önbellekteki temiz akciğer görüntüleri olduğundan, ön işleme hattı **çok daha sade** olur: artık `CenterCrop` yok (PA/AP köşe farkı zaten maskelendi), `CornerLetterMask` yok (R/L harfleri zaten silindi), RRR yok. Akciğer RoI'yi çerçevenin merkezinde tuttuğundan, artırma da **akciğeri kadrajdan kaçırmayacak** kadar yumuşak seçilir (hafif döndürme, yatay çevirme, küçük öteleme, ölçülü parlaklık/kontrast). Agresif `RandomResizedCrop` kasıtlı olarak kullanılmaz; aksi halde patolojik akciğer dokusunu kırpabilirdi.

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ]),
    'eval': transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(CONFIG["mean"], CONFIG["std"]),
    ]),
}

image_datasets = {
    'train': datasets.ImageFolder(os.path.join(CACHE_ROOT, 'train'), data_transforms['train']),
    'val'  : datasets.ImageFolder(os.path.join(CACHE_ROOT, 'val'),   data_transforms['eval']),
    'test' : datasets.ImageFolder(os.path.join(CACHE_ROOT, 'test'),  data_transforms['eval']),
}
dataloaders = {
    k: DataLoader(ds, batch_size=CONFIG["batch_size"], shuffle=(k == 'train'),
                  num_workers=2, pin_memory=True)
    for k, ds in image_datasets.items()
}

# ImageFolder sinif sirasi tutarli mi? (NORMAL=0, PNEUMONIA=1)
assert image_datasets['train'].classes == CLASS_NAMES, "Sinif sirasi uyusmazligi!"

print("="*56)
print("  ONBELLEKLENMIS (AKCIGER-ODAKLI) VERI")
print("="*56)
for k in ['train', 'val', 'test']:
    ds = image_datasets[k]
    c = Counter(l for _, l in ds.samples)
    print(f"  {k.upper():<6}: {len(ds):>5} | NORMAL:{c[0]:>4}  PNEUMONIA:{c[1]:>4}")
print("="*56)

# --- Nihai girdi izgarasi (sanity) ---
def denorm(t):
    x = t.clone()
    for i, (m, s) in enumerate(zip(CONFIG["mean"], CONFIG["std"])):
        x[i] = x[i]*s + m
    return x.clamp(0, 1)

imgs, lbls = next(iter(DataLoader(image_datasets['test'], batch_size=8, shuffle=True)))
fig, axes = plt.subplots(1, 8, figsize=(22, 3))
fig.suptitle('Modelin gordugu nihai girdiler (test) — yalnizca akciger', fontsize=12, fontweight='bold')
for ax, img, lbl in zip(axes, imgs, lbls):
    ax.imshow(denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(IDX_TO_CLASS[lbl.item()], fontsize=9, color=CLASS_COLORS[IDX_TO_CLASS[lbl.item()]])
    ax.axis('off')
plt.tight_layout()
plt.savefig('fig_02_input_grid.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. ViT-B/16 Mimarisi ve Eğitim <a id='8'></a>

Sınıflandırıcı, ImageNet ön-eğitimli **ViT-B/16**'dır. Gövde dondurulur; yalnızca son `fine_tune_last_n` Transformer bloğu, son katman normalizasyonu ve yeni sınıflandırma başlığı eğitilir. Bu, küçük tıbbi veride aşırı uyumu (overfitting) sınırlar.

**Kayıp fonksiyonu artık sadece CrossEntropy'dir.** Önceki sürümdeki `AnatomySuppressionHook` ve RRR gradyan-cezası terimi **tamamen kaldırılmıştır** — çünkü "akciğer dışına bakma" kısıtı artık bir ceza ile değil, girdinin kendisiyle (akciğer dışı bilgi mevcut değil) sağlanmaktadır. Bu, eğitimi hem sadeleştirir hem hızlandırır.

In [ ]:
def build_vit(fine_tune_last_n=2):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.encoder.layers[-fine_tune_last_n:].parameters():
        p.requires_grad = True
    for p in model.encoder.ln.parameters():
        p.requires_grad = True
    in_f = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.Dropout(CONFIG["dropout"]),
        nn.Linear(in_f, CONFIG["num_classes"]),
    )
    return model.to(device)

model = build_vit(fine_tune_last_n=2)
tot = sum(p.numel() for p in model.parameters())
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Toplam parametre: {tot:,} | Egitilebilir: {trn:,} ({trn/tot:.1%})")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    loss_sum, preds, labels, probs = 0.0, [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            if train:
                optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            loss_sum += loss.item() * imgs.size(0)
            probs.extend(torch.softmax(out, 1)[:, 1].detach().cpu().numpy())
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(lbls.cpu().numpy())
    N = len(loader.dataset)
    acc = (np.array(preds) == np.array(labels)).mean()
    return loss_sum/N, f1_score(labels, preds, zero_division=0), acc, labels, preds, probs

print("="*68)
print(f"  AKCIGER-ODAKLI ViT EGITIMI | batch={CONFIG['batch_size']} | {CONFIG['epochs']} epoch | sadece CE")
print("="*68)

history = {k: [] for k in ['tr_loss', 'tr_f1', 'vl_loss', 'vl_f1', 'vl_acc']}
best_f1, best_wt = 0.0, None

for epoch in range(CONFIG["epochs"]):
    tr_loss, tr_f1, _, _, _, _      = run_epoch(model, dataloaders['train'], criterion, optimizer)
    vl_loss, vl_f1, vl_acc, _, _, _ = run_epoch(model, dataloaders['val'],   criterion, None)
    scheduler.step()
    for k, v in zip(['tr_loss','tr_f1','vl_loss','vl_f1','vl_acc'],
                    [tr_loss, tr_f1, vl_loss, vl_f1, vl_acc]):
        history[k].append(v)
    star = ''
    if vl_f1 > best_f1:
        best_f1 = vl_f1
        best_wt = {k: v.clone() for k, v in model.state_dict().items()}
        star = ' *'
    print(f"Ep{epoch+1:3d} | TR loss {tr_loss:.4f} f1 {tr_f1:.4f} | "
          f"VAL loss {vl_loss:.4f} f1 {vl_f1:.4f} acc {vl_acc:.4f}{star}")

if best_wt:
    model.load_state_dict(best_wt)
print(f"\nEn iyi Val F1 = {best_f1:.4f} yuklendi.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Akciger-Odakli ViT — Egitim Gecmisi (RRR yok, saf CE)', fontsize=13, fontweight='bold')
ep = range(1, len(history['tr_loss'])+1)

ax = axes[0]
ax.plot(ep, history['tr_loss'], 'o-', color='#2196F3', lw=2, ms=4, label='Egitim Kaybi')
ax.plot(ep, history['vl_loss'], 's--', color='#F44336', lw=2, ms=4, label='Val Kaybi')
ax.set_title('Kayip'); ax.set_xlabel('Epoch'); ax.legend(); sns.despine(ax=ax)

ax = axes[1]
ax.plot(ep, history['tr_f1'], 'o-', color='#4CAF50', lw=2, ms=4, label='Egitim F1')
ax.plot(ep, history['vl_f1'], 's--', color='#FF9800', lw=2, ms=4, label='Val F1')
bi = int(np.argmax(history['vl_f1']))
ax.axvline(bi+1, color='purple', ls=':', lw=1.5, label=f'En iyi (Ep{bi+1})')
ax.set_title('F1 Skoru'); ax.set_xlabel('Epoch'); ax.legend(); sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('fig_03_training.png', dpi=130, bbox_inches='tight')
plt.show()

## 9. Model Değerlendirmesi <a id='9'></a>

Model, dengeli test kümesinde değerlendirilir. Confusion matrix, ROC ve Precision-Recall eğrileri birlikte raporlanır. Tıbbi tarama bağlamında **Recall (duyarlılık)** özellikle önemlidir: kaçırılan bir pnömoni (yanlış negatif), gereksiz bir alarmdan (yanlış pozitif) klinik olarak daha maliyetlidir.

In [ ]:
test_loss, test_f1, test_acc, y_true, y_pred, y_probs = run_epoch(model, dataloaders['test'], criterion, None)
print(f"  Test Kaybi    : {test_loss:.4f}")
print(f"  Test Dogruluk : {test_acc:.4f} ({test_acc:.1%})")
print(f"  Test F1       : {test_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

fig = plt.figure(figsize=(20, 6))
gs  = gridspec.GridSpec(1, 3, wspace=0.30)

ax1 = fig.add_subplot(gs[0])
cm = confusion_matrix(y_true, y_pred)
cm_n = cm.astype(float) / cm.sum(1)[:, None]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='white', cbar=False, annot_kws={'size': 14, 'weight': 'bold'})
for i in range(2):
    for j in range(2):
        ax1.text(j+0.5, i+0.72, f'({cm_n[i, j]:.1%})', ha='center', fontsize=9, color='gray')
ax1.set_title('Confusion Matrix', fontweight='bold'); ax1.set_xlabel('Tahmin'); ax1.set_ylabel('Gercek')

ax2 = fig.add_subplot(gs[1])
fpr, tpr, _ = roc_curve(y_true, y_probs); roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='#2196F3', lw=2.5, label=f'AUC = {roc_auc:.4f}')
ax2.plot([0, 1], [0, 1], 'k--', lw=1); ax2.fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR'); ax2.set_title('ROC Egrisi', fontweight='bold')
ax2.legend(); sns.despine(ax=ax2)

ax3 = fig.add_subplot(gs[2])
prec, rec, thr = precision_recall_curve(y_true, y_probs); ap = average_precision_score(y_true, y_probs)
f1s = 2*(prec*rec)/(prec+rec+1e-8); bi = int(np.argmax(f1s[:-1]))
ax3.plot(rec, prec, color='#4CAF50', lw=2.5, label=f'AP = {ap:.4f}')
ax3.scatter(rec[bi], prec[bi], color='red', s=120, zorder=5, label=f'En iyi esik = {thr[bi]:.3f}')
ax3.fill_between(rec, prec, alpha=0.1, color='#4CAF50')
ax3.set_xlabel('Recall'); ax3.set_ylabel('Precision'); ax3.set_title('Precision-Recall', fontweight='bold')
ax3.legend(); sns.despine(ax=ax3)

fig.suptitle('Akciger-Odakli ViT — Test Seti Degerlendirmesi', fontsize=14, fontweight='bold')
plt.savefig('fig_04_eval.png', dpi=130, bbox_inches='tight')
plt.show()

## 10. Açıklanabilirlik: Attention Rollout <a id='10'></a>

Modelin yalnızca akciğere baktığı *yapısal olarak* garanti edilse de, **akciğerin neresine** baktığı hâlâ önemlidir: ideal model, parankimdeki konsolidasyon/infiltrasyon bölgelerine odaklanmalıdır. Bunu görmek için **Attention Rollout** (Abnar & Zuidema, 2020) kullanılır: tüm katmanlardaki dikkat matrisleri, artık (residual) bağlantılar hesaba katılarak çarpılır ve CLS token'ından patch'lere akan toplam dikkat elde edilir.

> Grad-CAM, önceki sürümde RRR cezasıyla etkileşince yanıltıcı bir "Silgi Etkisi" üretmişti; RRR kaldırıldığı için bu sorun ortadan kalkmıştır, yine de bilgi akışını en sadık izleyen yöntem olduğundan Attention Rollout tercih edilir.

In [ ]:
class AttentionRollout:
    """ViT katmanlarindan dikkat matrislerini toplar; residual + normalize ile CLS->patch akisi."""
    def __init__(self, model):
        self.model = model
        self.attn_maps = []
        self._hooks = []
        for layer in self.model.encoder.layers:
            sa = layer.self_attention
            orig = sa.forward
            def make_patched(orig_fwd):
                def patched(self_mod, *a, **kw):
                    kw['need_weights'] = True
                    kw['average_attn_weights'] = False  # kafalari erken ortalama; (B,H,L,L) dondur
                    return orig_fwd(*a, **kw)
                return patched
            sa.forward = types.MethodType(make_patched(orig), sa)
            def make_hook(obj):
                def hook(m, inp, out):
                    if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
                        obj.attn_maps.append(out[1].detach().cpu())
                return hook
            self._hooks.append(sa.register_forward_hook(make_hook(self)))

    def remove(self):
        for h in self._hooks:
            h.remove()

    def compute(self, img_tensor, head_fusion='mean'):
        self.attn_maps = []
        self.model.eval()
        with torch.no_grad():
            self.model(img_tensor.unsqueeze(0).to(device))
        if not self.attn_maps:
            raise RuntimeError("Attention yakalanamadi.")
        N = self.attn_maps[0].size(-1)
        result = torch.eye(N)
        for attn in self.attn_maps:
            # average_attn_weights=False ise attn: (B,H,L,L); aksi halde (B,L,L)
            a = attn.squeeze(0)                       # (H,L,L) veya (L,L)
            if a.dim() == 3:                          # kafa ekseni mevcut -> fuze et
                a = a.mean(0) if head_fusion == 'mean' else a.max(0)[0]
            # buradan sonra a kesinlikle (L,L)
            a = a + torch.eye(N)
            a = a / (a.sum(-1, keepdim=True) + 1e-8)
            result = torch.matmul(a, result)
        mask = result[0, 1:].numpy()
        n = int(round(mask.size ** 0.5))
        m14 = mask.reshape(n, n)
        m14 = (m14 - m14.min()) / (m14.max() - m14.min() + 1e-8)
        m224 = cv2.resize(m14.astype(np.float32), (CONFIG["img_size"], CONFIG["img_size"]))
        m224 = (m224 - m224.min()) / (m224.max() - m224.min() + 1e-8)
        return m224, m14

try: rollout.remove()
except: pass
rollout = AttentionRollout(model)
print(f"Attention Rollout hazir — {len(rollout._hooks)} katman izleniyor.")

In [ ]:
def overlay_heatmap(img_t, heat, alpha=0.45):
    img = denorm(img_t).permute(1, 2, 0).numpy()
    col = MEDICAL_CMAP(heat)[:, :, :3]
    return np.clip((1-alpha)*img + alpha*col, 0, 1), img

# Her siniftan birkac ornek
N_PER = 3
eval_ds = image_datasets['test']
sel = {c: [] for c in CLASS_NAMES}
order = list(range(len(eval_ds.samples)))
random.Random(SEED+1).shuffle(order)
for i in order:
    p, l = eval_ds.samples[i]
    c = IDX_TO_CLASS[l]
    if len(sel[c]) < N_PER:
        sel[c].append(i)
    if all(len(v) >= N_PER for v in sel.values()):
        break

rows = sum(len(v) for v in sel.values())
fig, axes = plt.subplots(rows, 3, figsize=(11, 3.6*rows))
fig.suptitle('Attention Rollout — modelin akciger-ICI odagi', fontsize=12, fontweight='bold', y=1.0)
r = 0
for c in CLASS_NAMES:
    for i in sel[c]:
        img_t, _ = eval_ds[i]
        m224, _ = rollout.compute(img_t)
        ov, img = overlay_heatmap(img_t, m224)
        axes[r, 0].imshow(img); axes[r, 0].set_title(f'{c} — girdi', fontsize=9,
                          color=CLASS_COLORS[c], fontweight='bold'); axes[r, 0].axis('off')
        axes[r, 1].imshow(m224, cmap=MEDICAL_CMAP); axes[r, 1].set_title('Attention (224)', fontsize=9); axes[r, 1].axis('off')
        axes[r, 2].imshow(ov); axes[r, 2].set_title('Bindirme', fontsize=9); axes[r, 2].axis('off')
        r += 1
plt.tight_layout()
plt.savefig('fig_05_xai.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Akciğer-Odak Doğrulaması (LFR) <a id='11'></a>

$S(u)\in[0,1]$ normalize edilmiş Attention Rollout değeri, $L(u)\in\{0,1\}$ ise modele verilen akciğer maskesi (§6'da önbelleğe alınan) olsun. Gürültü eşiği $\tau_n=0.2$ ile yüksek-enerjili piksel kümesi $\Omega=\{u:S(u)>\tau_n\}$ tanımlanır. LFR, yüksek-enerjili dikkatin akciğer içinde kalan oranıdır:

$$\mathrm{LFR}=\frac{\sum_{u\in\Omega} S(u)\,L(u)}{\sum_{u\in\Omega} S(u)},\qquad \mathrm{LFR}\in[0,1].$$

**Yorumun dürüst çerçevesi.** Önceki (RRR'li) sürümde LFR güçlü bir testti, çünkü model tüm CXR'ı görüyordu ve akciğer dışına bakmamayı *seçmek* zorundaydı; LFR≈0.16 bu seçimi yapmadığını ifşa etmişti. Bu sürümde girdi zaten yalnızca akciğerden ibarettir, dolayısıyla yüksek LFR kısmen **tasarım gereğidir** ("right reasons by construction"). Burada LFR, daha mütevazı ama yine de anlamlı bir soruyu yanıtlar: *model dikkatini görünür akciğer dokusuna mı veriyor, yoksa maskelenmiş (bilgisiz) kenar/dolgu bölgesine mi israf ediyor?* Yüksek LFR, modelin akciğer sinyalini etkin kullandığını; düşük LFR ise dikkatin hâlâ kenar/dolgu artefaktına kaydığını gösterir.

In [ ]:
def lung_focus_ratio(saliency, lung_mask, tau=0.2):
    s = np.asarray(saliency, dtype=np.float32).flatten()
    m = np.asarray(lung_mask, dtype=np.float32).flatten()
    hi = s > tau
    if not hi.any():
        return float('nan')
    tot = s[hi].sum()
    if tot < 1e-8:
        return float('nan')
    return float((s[hi]*m[hi]).sum() / tot)

def load_cached_mask(split, cls, stem224):
    """Onbellekteki 224 akciger maskesini yukler (LFR icin)."""
    p = os.path.join(CACHE_ROOT, f"{split}_mask", cls, os.path.basename(stem224))
    m = np.asarray(Image.open(p).convert('L'))
    return (m > 127).astype(np.float32)

MAX_PER_SPLIT = 60
splits = ['train', 'val', 'test']
agg = {sp: {c: {'cam': np.zeros((224, 224)), 'mask': np.zeros((224, 224)), 'n': 0, 'lfr': []}
            for c in CLASS_NAMES} for sp in splits}

for sp in splits:
    ds = datasets.ImageFolder(os.path.join(CACHE_ROOT, sp), data_transforms['eval'])
    order = list(range(len(ds.samples))); random.Random(SEED).shuffle(order)
    cnt = 0
    for i in order:
        if cnt >= MAX_PER_SPLIT*2:   # her iki sinif birlikte
            break
        path, lbl = ds.samples[i]; cls = IDX_TO_CLASS[lbl]
        if agg[sp][cls]['n'] >= MAX_PER_SPLIT:
            continue
        try:
            img_t, _ = ds[i]
            sal, _ = rollout.compute(img_t)
            mask = load_cached_mask(sp, cls, path)
            lfr = lung_focus_ratio(sal, mask, tau=0.2)
            if np.isnan(lfr):
                continue
            agg[sp][cls]['cam']  += sal
            agg[sp][cls]['mask'] += mask
            agg[sp][cls]['n']    += 1
            agg[sp][cls]['lfr'].append(lfr)
            cnt += 1
        except Exception:
            continue
    msg = " | ".join(f"{c}: n={agg[sp][c]['n']} LFR={np.mean(agg[sp][c]['lfr']):.3f}"
                     for c in CLASS_NAMES if agg[sp][c]['n'] > 0)
    print(f"[{sp.upper():<5}] {msg}")

In [ ]:
# --- Global aggregate gorsel ---
g_cam = np.zeros((224, 224)); g_mask = np.zeros((224, 224)); g_lfr = []; g_n = 0
for sp in splits:
    for c in CLASS_NAMES:
        d = agg[sp][c]
        g_cam += d['cam']; g_mask += d['mask']; g_lfr.extend(d['lfr']); g_n += d['n']

g_lfr = np.array(g_lfr)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Akciger-Odak Dogrulamasi — Aggregate Attention + LFR', fontsize=13, fontweight='bold')

cam_avg = g_cam / max(g_n, 1)
cam_avg = (cam_avg - cam_avg.min()) / (cam_avg.max() - cam_avg.min() + 1e-8)
axes[0].imshow(cam_avg, cmap=MEDICAL_CMAP)
axes[0].set_title(f'Genel Aggregate Attention (n={g_n})', fontsize=11, fontweight='bold'); axes[0].axis('off')

axes[1].imshow(cam_avg, cmap=MEDICAL_CMAP)
axes[1].contour(g_mask/max(g_n, 1), levels=[0.4], colors='lime', linewidths=2.0)
axes[1].set_title(f'+ Gorunur akciger konturu\nGlobal LFR = {g_lfr.mean():.3f} +/- {g_lfr.std():.3f}',
                  fontsize=11, fontweight='bold'); axes[1].axis('off')

ax = axes[2]
for sp, col in zip(splits, ['#4C72B0', '#DD8452', '#55A868']):
    vals = [v for c in CLASS_NAMES for v in agg[sp][c]['lfr']]
    if vals:
        ax.hist(vals, bins=16, alpha=0.55, color=col, edgecolor='white',
                density=True, label=f'{sp.upper()} (mu={np.mean(vals):.2f})')
ax.axvline(0.65, color='green', ls='--', lw=2, label='Iyi esik 0.65')
ax.set_xlabel('Lung Focus Ratio'); ax.set_ylabel('Yogunluk'); ax.set_xlim(0, 1)
ax.set_title('Split bazli LFR dagilimi', fontsize=11, fontweight='bold'); ax.legend(fontsize=9)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('fig_06_lung_focus.png', dpi=130, bbox_inches='tight')
plt.show()

print("\n" + "="*58)
print("  AKCIGER-ODAK DOGRULAMASI — SONUC")
print("="*58)
print(f"  Analiz edilen goruntu : {g_n}")
print(f"  Global LFR            : {g_lfr.mean():.3f} +/- {g_lfr.std():.3f}  (median {np.median(g_lfr):.3f})")
if g_lfr.mean() >= 0.65:
    print("  -> Dikkat agirlikli olarak gorunur akciger dokusunda.")
elif g_lfr.mean() >= 0.50:
    print("  -> Akcigere yonelim var ama kenar/dolguda da enerji mevcut.")
else:
    print("  -> Dikkat hala kenar/dolgu bolgesine kayiyor (maske payi/feather gozden gecirilebilir).")
print("  NOT: Girdi zaten maskeli oldugundan LFR kismen tasarim geregidir (bkz. 11. bolum).")
print("="*58)

## Modeli Kaydetme — Kaggle'a Yükleme ve Dış Doğrulama için <a id='save'></a>

Eğitilen model (yukarıda **en iyi Val F1** ağırlıkları yüklenmiş haliyle) tek bir kontrol noktası olarak `/kaggle/working/` altına yazılır. Kontrol noktası yalnızca ağırlıkları değil, **dış doğrulamada ön-işleme hattını birebir yeniden kurmak için gereken her şeyi** içerir: mimari adı, sınıf eşlemesi ve tüm maskeleme/ölçekleme parametreleri (`CONFIG`). Böylece doğrulama notebook'u aynı `mean/std`, aynı maske payı ve aynı RoI mantığını garanti eder.

**Kaggle'a yükleme adımları.**
1. Bu notebook'u baştan sona çalıştırın; sağdaki **Output** sekmesinde `lung_focused_vit_pneumonia.pth` belirir.
2. Dosyayı indirin (veya "New Dataset" ile doğrudan output'tan dataset oluşturun).
3. **Yeni bir Kaggle Dataset** oluşturup bu dosyayı yükleyin. Önerilen ad: **`lung-focused-vit-pneumonia`**.
4. Dış doğrulama notebook'u dosyayı bu adla (ya da herhangi bir `.pth`) `/kaggle/input/` altında **otomatik** bulur.

In [ ]:
import json

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)
ckpt_path = os.path.join(SAVE_DIR, "lung_focused_vit_pneumonia.pth")

checkpoint = {
    "model_state_dict" : model.state_dict(),   # en iyi Val F1 agirliklari yuklu
    "arch"             : "vit_b_16",
    "fine_tune_last_n" : 2,
    "num_classes"      : CONFIG["num_classes"],
    "class_names"      : CLASS_NAMES,           # ['NORMAL','PNEUMONIA'] -> 0,1
    "class_to_idx"     : CLASS_TO_IDX,
    "config"           : CONFIG,                # tum on-isleme + maskeleme parametreleri
    "best_val_f1"      : float(best_f1),
    "input_pipeline"   : ("ianpan_lung_mask(R+L, kalp haric) -> dilation+feather+mean_fill "
                          "-> RoI crop -> Resize224 -> Normalize(mean,std)"),
}
torch.save(checkpoint, ckpt_path)

# Insan denetimi icin CONFIG'i okunabilir JSON olarak da yaz
with open(os.path.join(SAVE_DIR, "lung_focused_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

mb = os.path.getsize(ckpt_path) / 1e6
print("="*58)
print("  MODEL KAYDEDILDI")
print("="*58)
print(f"  Dosya        : {ckpt_path}")
print(f"  Boyut        : {mb:.1f} MB")
print(f"  En iyi Val F1: {best_f1:.4f}")
print(f"  Siniflar     : {CLASS_TO_IDX}")
print("="*58)
print("  -> Output panelinden indirip 'lung-focused-vit-pneumonia'")
print("     adiyla yeni bir Kaggle Dataset olarak yukleyin.")
print("="*58)

## 12. Tartışma ve Sonuç <a id='12'></a>

### 12.1. Yöntemlerin Karşılaştırması

| Boyut | Önceki: Yumuşak RRR Cezası | Bu çalışma: Sert Girdi Maskeleme |
|-------|----------------------------|----------------------------------|
| Kısıtın doğası | Türevlenebilir kayıp terimi ($\lambda \cdot L_{\text{anatomi}}$) | Veri hattının yapısı (akciğer dışı bilgi yok) |
| Kısayol dayanıklılığı | **Dolanılabilir** — model cezayı küçültürken omurga/mediasten kısayolunu sürdürdü (LFR≈0.16) | **Fiziksel olarak imkânsız** — omurga/mediasten/köşe girdide yok |
| Eğitim maliyeti | Her batch'te canlı segmentasyon + gradyan cezası (yavaş) | Tek seferlik offline maskeleme; eğitim normal hızda |
| Karmaşıklık | Hook + ceza ayarı ($\lambda$ taraması) | Saf CrossEntropy |
| Bağımlılık | Segmentasyon + ceza dengesi | Yalnızca segmentasyon kalitesi |

### 12.2. Temel Sonuç

Önceki çalışmanın dersi — *yumuşak bir cezanın güçlü bir kısayolu caydıramaması* — bu sürümü doğrudan motive etmiştir. Akciğer dışındaki bilgiyi girdi seviyesinde silmek, **omurga/mediasten kısayolunu, köşe artefaktlarını ve R/L işaretleyicilerini tek bir hamlede ortadan kaldırır.** *Right for the Right Reasons*, artık eğitimin iyi niyetine değil, mimarinin yapısına bağlıdır.

### 12.3. Kısıtlar ve Dürüst Değerlendirme

1. **Segmentasyon tavan etkisi.** Modelin doğruluğu artık ianpan'ın akciğer sınırını doğru çizmesine bağımlıdır. Dice R=0.957 / L=0.948 yüksek olsa da kusursuz değildir; ağır konsolidasyon veya masif efüzyonda akciğer sınırı silikleşir ve maske patolojinin bir kısmını dışarıda bırakabilir. Maske genişletme + RoI payı + kenar yumuşatma bu riski azaltır ama sıfırlamaz. **Öneri:** birkaç fallback/düşük-maske vakasını elle denetleyin.
2. **Kasıtlı bağlam kaybı.** Kardiyotorasik oran, mediastinal genişlik gibi akciğer-dışı klinik ipuçları kasıtlı olarak atılır. Bu, kısayolsuzluk uğruna ödenen bilinçli bir bedeldir; yöntem "tam radyolojik okuma" değil, **parankim-temelli, kısayoldan arındırılmış** bir sınıflandırmadır.
3. **LFR'nin yumuşaması.** §11'de tartışıldığı gibi, maskeli girdide yüksek LFR kısmen totolojiktir; bu metrik artık "akciğeri buldu mu" değil "akciğer-içi dikkati nereye verdi / kenar artefaktına kaçtı mı" sorusunu ölçer.
4. **Postürün akciğer-içi izi.** Maskeleme, postür ipuçlarını *akciğer dışında* (gövde dokusu, köşe) yok eder; ancak postür, akciğer *şeklini* de etkilediğinden (AP'de daha kısa/geniş projeksiyon) tümüyle elenmemiş olabilir. ianpan'ın `out["view"]` çıktısıyla AP/PA dağılımını sınıf-bazlı raporlamak, kalan postür yanlılığını ölçmek için iyi bir sonraki adımdır.

### 12.4. Gelecek Çalışmalar

- **Segmentasyon duyarlılık analizi:** maske payını (`mask_dilate_frac`, `roi_pad_frac`) tarayıp F1 üzerindeki etkisini ölçmek; çok dar maskenin patoloji kaybına, çok geniş maskenin kısayol sızıntısına yol açtığı Pareto noktasını bulmak.
- **Postür dengeleme:** `out["view"]` (AP/PA/lateral) meta-verisiyle katmanlı örnekleme veya domain-adversarial eğitim.
- **Dış-dağıtım doğrulama:** farklı merkez/cihazdan harici test kümesinde dağıtım kayması altında çöküşü ölçmek (kısayolsuzluğun asıl sınavı).
- **Yumuşak vs sert karşılaştırması:** aynı veri/bölme üzerinde RRR'li ve maskeli modellerin LFR ve dış-dağıtım F1'ini doğrudan kıyaslamak.

> **Klinik Not.** Yöntem akciğer-dışı bağlamı attığından ve segmentasyona bağımlı olduğundan, **klinik dağıtım için hazır değildir.** Dikkat haritalarının ve özellikle düşük-maske/fallback vakalarının uzman radyolog denetiminden geçmesi zorunludur.

### Referanslar

1. Ross, A. S., Hughes, M. C., & Doshi-Velez, F. (2017). *Right for the Right Reasons.* IJCAI.
2. Abnar, S., & Zuidema, W. (2020). *Quantifying Attention Flow in Transformers.* ACL.
3. DeGrave, A. J., Janizek, J. D., & Lee, S.-I. (2021). *AI for radiographic COVID-19 detection selects shortcuts over signal.* Nature Machine Intelligence.
4. Geirhos, R., et al. (2020). *Shortcut Learning in Deep Neural Networks.* Nature Machine Intelligence.
5. Dosovitskiy, A., et al. (2021). *An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale.* ICLR.
6. Gaggion, N., et al. (2024). *CheXmask: a large-scale dataset of anatomical segmentation masks for multi-center chest x-ray images.* Scientific Data. (ianpan/chest-x-ray-basic'in eğitim verisi)

---
*Akciğer-Odaklı ViT · Anatomik Girdi Maskeleme · ianpan U-Net · Attention Rollout · LFR Doğrulaması*